### Step 1: Base model (tidied)

This is the team's original base model with small cleanups only. The model, data, and results are unchanged.
See the **Changes from the original** list at the bottom of this notebook.

You work at a media company, and you want to sell advertising spots for the upcoming NFL season. You project ratings for the prospective customers, and price the package based on the ratings.

**4 ratings projection packages:**

| Package | Projected viewers | Price | CPM (cost per thousand viewers) |
| --- | ---: | ---: | ---: |
| conservative | 1,800,000 | \$90,000 | \$50.00 |
| medium | 2,100,000 | \$100,000 | \$47.62 |
| premium | 2,500,000 | \$110,000 | \$44.00 |
| aggressive | 3,000,000 | \$125,000 | \$41.67 |

CPM = price / (projected viewers / 1,000). For example, \$90,000 / 1,800 = \$50.00.

**How many viewers will actually watch?**

Management gives the probability of reaching *at least* each projection:

| Actual viewers | P(at least this many) |
| --- | ---: |
| 1,800,000 or more | 80% |
| 2,100,000 or more | 60% |
| 2,500,000 or more | 45% |
| 3,000,000 or more | 35% |

These are cumulative ("at least") probabilities. They overlap, so they add up to more than 100% and can't be used directly as scenario probabilities.
Subtracting neighboring levels gives the probability of landing in each range, which does add up to 100%:

| Actual viewers | Probability | Value used in the model |
| --- | ---: | ---: |
| Below 1.8M | 100% - 80% = 20% | 1,500,000 |
| 1.8M to 2.1M | 80% - 60% = 20% | 1,800,000 |
| 2.1M to 2.5M | 60% - 45% = 15% | 2,100,000 |
| 2.5M to 3.0M | 45% - 35% = 10% | 2,500,000 |
| 3.0M or more | 35% | 3,000,000 |

**Assumptions:**
1. 1,500,000 is a chosen stand-in value for the "below 1.8M" range.
2. Each range is valued at its lower end (for example, anything from 1.8M to 2.1M counts as 1.8M). This slightly overstates the expected penalty, which is the cautious choice.

**Prospects and penalties**

Next, you have your prospects list, including their name, their goal CPM, and their underdelivery penalty. A prospect will only buy a package whose CPM is at or below their goal CPM. If you sell a package to a customer and don't deliver the projected viewers, you owe them the difference between the projection and the actual viewers, times their underdelivery penalty (dollars per missing viewer).

You have 20 ad spots to sell. You need to maximize expected profit: revenue from the packages sold, minus the expected underdelivery penalty, weighted by the probability of each viewer outcome.

| ID  | Prospect               | Goal CPM | Underdelivery Penalty |
| --- | ---------------------- | -------: | --------------------: |
| C01 | Car Dealer A           |    42.71 |                  0.05 |
| C02 | Car Dealer B           |    43.33 |                  0.15 |
| C03 | Grocery Store A        |    40.24 |                  0.10 |
| C04 | Specialty Retail A     |    43.95 |                  0.05 |
| C05 | Fast Food A            |    44.57 |                  0.05 |
| C06 | Fast Food B            |    45.19 |                  0.25 |
| C07 | Grocery Store B        |    45.80 |                  0.05 |
| C08 | Car Dealer C           |    46.43 |                  0.15 |
| C09 | Fast Food C            |    40.85 |                  0.10 |
| C10 | Sporting Goods A       |    47.04 |                  0.05 |
| C11 | Farm Equipment A       |    47.67 |                  0.10 |
| C12 | Home Improvement A     |    48.28 |                  0.25 |
| C13 | Bank A                 |    48.90 |                  0.05 |
| C14 | Furniture Store A      |    49.52 |                  0.20 |
| C15 | Auto Parts A           |    41.48 |                  0.25 |
| C16 | Insurance Agency A     |    50.14 |                  0.20 |
| C17 | HVAC Company A         |    50.76 |                  0.05 |
| C18 | Restaurant A           |    51.38 |                  0.10 |
| C19 | Farm Supply A          |    51.99 |                  0.10 |
| C20 | Credit Union A         |    52.62 |                  0.05 |
| C21 | Outdoor Recreation A   |    53.23 |                  0.20 |
| C22 | Beverage Distributor A |    42.09 |                  0.10 |
| C23 | Car Dealer D           |    53.86 |                  0.25 |
| C24 | Grocery Store C        |    54.47 |                  0.20 |
| C25 | Home Builder A         |    55.09 |                  0.15 |
| C26 | Healthcare System A    |    55.71 |                  0.25 |
| C27 | Wireless Retailer A    |    56.33 |                  0.25 |
| C28 | Convenience Store A    |    42.40 |                  0.25 |
| C29 | Ag Services A          |    57.57 |                  0.15 |
| C30 | Financial Services A   |    58.81 |                  0.20 |

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import course_utils  # course helpers (print_status, print_model, ...)

In [ ]:
# ============================================================
# 1. INDEX SETS
# ============================================================

# packages are the 4 ratings projections we can sell.
# (The original called these "scenarios". Renamed so they aren't
# confused with the actual-viewer outcomes in section 4.)

customers = [
    "C01", "C02", "C03", "C04", "C05",
    "C06", "C07", "C08", "C09", "C10",
    "C11", "C12", "C13", "C14", "C15",
    "C16", "C17", "C18", "C19", "C20",
    "C21", "C22", "C23", "C24", "C25",
    "C26", "C27", "C28", "C29", "C30"
]

packages = [
    "conservative",
    "medium",
    "premium",
    "aggressive"
]


# ============================================================
# 2. CUSTOMER PARAMETERS
# ============================================================

goal_cpm = {
    "C01": 42.71,
    "C02": 43.33,
    "C03": 40.24,
    "C04": 43.95,
    "C05": 44.57,
    "C06": 45.19,
    "C07": 45.80,
    "C08": 46.43,
    "C09": 40.85,
    "C10": 47.04,
    "C11": 47.67,
    "C12": 48.28,
    "C13": 48.90,
    "C14": 49.52,
    "C15": 41.48,
    "C16": 50.14,
    "C17": 50.76,
    "C18": 51.38,
    "C19": 51.99,
    "C20": 52.62,
    "C21": 53.23,
    "C22": 42.09,
    "C23": 53.86,
    "C24": 54.47,
    "C25": 55.09,
    "C26": 55.71,
    "C27": 56.33,
    "C28": 42.40,
    "C29": 57.57,
    "C30": 58.81
}

underdelivery_penalty = {
    "C01": 0.05,
    "C02": 0.15,
    "C03": 0.10,
    "C04": 0.05,
    "C05": 0.05,
    "C06": 0.25,
    "C07": 0.05,
    "C08": 0.15,
    "C09": 0.10,
    "C10": 0.05,
    "C11": 0.10,
    "C12": 0.25,
    "C13": 0.05,
    "C14": 0.20,
    "C15": 0.25,
    "C16": 0.20,
    "C17": 0.05,
    "C18": 0.10,
    "C19": 0.10,
    "C20": 0.05,
    "C21": 0.20,
    "C22": 0.10,
    "C23": 0.25,
    "C24": 0.20,
    "C25": 0.15,
    "C26": 0.25,
    "C27": 0.25,
    "C28": 0.25,
    "C29": 0.15,
    "C30": 0.20
}


# ============================================================
# 3. AD PACKAGE PARAMETERS
# ============================================================

views = {
    "conservative": 1_800_000,
    "medium": 2_100_000,
    "premium": 2_500_000,
    "aggressive": 3_000_000
}

price = {
    "conservative": 90_000,
    "medium": 100_000,
    "premium": 110_000,
    "aggressive": 125_000
}

cpm = {
    "conservative": 50.00,
    "medium": 47.62,
    "premium": 44.00,
    "aggressive": 41.67
}


# ============================================================
# 4. ACTUAL VIEWER OUTCOME DISTRIBUTION
# ============================================================

# Management's probabilities are cumulative ("at least"):
#
# P(Viewers >= 1.8M) = 80%
# P(Viewers >= 2.1M) = 60%
# P(Viewers >= 2.5M) = 45%
# P(Viewers >= 3.0M) = 35%
#
# Subtracting neighboring levels gives the probability
# of landing in each range:
#
# Below 1.8M     = 100% - 80% = 20%
# 1.8M - 2.1M    =  80% - 60% = 20%
# 2.1M - 2.5M    =  60% - 45% = 15%
# 2.5M - 3.0M    =  45% - 35% = 10%
# 3.0M+          =               35%
#
# Assumptions:
# - 1.5M stands in for the "below 1.8M" range.
# - Each range is valued at its lower end
#   (e.g., 1.8M - 2.1M counts as 1.8M).


actual_viewers = {
    1_500_000: 0.20,
    1_800_000: 0.20,
    2_100_000: 0.15,
    2_500_000: 0.10,
    3_000_000: 0.35
}


# Verify that probabilities equal 100%

assert abs(sum(actual_viewers.values()) - 1.0) < 0.000001


# ============================================================
# 5. DETERMINE PACKAGE ELIGIBILITY
# ============================================================

# eligible[c,p] = 1 if customer c's goal CPM is high enough
# to accept package p.
#
# Otherwise eligible[c,p] = 0.

eligible = {}

for c in customers:
    for p in packages:

        if cpm[p] <= goal_cpm[c]:
            eligible[c, p] = 1
        else:
            eligible[c, p] = 0


# ============================================================
# 6. CALCULATE EXPECTED UNDERDELIVERY PENALTY
# ============================================================

expected_penalty = {}

for c in customers:
    for p in packages:

        expected_penalty[c, p] = sum(
            probability *
            max(views[p] - actual, 0) *
            underdelivery_penalty[c]
            for actual, probability
            in actual_viewers.items()
        )


# ============================================================
# 7. CALCULATE EXPECTED PROFIT
# ============================================================

expected_profit = {}

for c in customers:
    for p in packages:

        expected_profit[c, p] = (
            price[p]
            - expected_penalty[c, p]
        )


# ============================================================
# 8. CREATE GUROBI MODEL
# ============================================================

model = gp.Model("NFL_Advertising_Optimization")


# ============================================================
# 9. DECISION VARIABLES
# ============================================================

# x[c,p] = 1 if customer c receives package p
#          0 otherwise

x = model.addVars(
    customers,
    packages,
    vtype=GRB.BINARY,
    name="x"
)


# ============================================================
# 10. CONSTRAINTS
# ============================================================

# ------------------------------------------------------------
# Constraint 1:
# Each customer can receive at most one package.
# ------------------------------------------------------------

model.addConstrs(
    (
        gp.quicksum(x[c, p] for p in packages) <= 1
        for c in customers
    ),
    name="one_package_per_customer"
)


# ------------------------------------------------------------
# Constraint 2:
# Package CPM cannot exceed customer's goal CPM.
# ------------------------------------------------------------

model.addConstrs(
    (
        x[c, p] <= eligible[c, p]
        for c in customers
        for p in packages
    ),
    name="cpm_requirement"
)


# ------------------------------------------------------------
# Constraint 3:
# We have 20 advertising spots, so we can sell at most 20 packages.
# ------------------------------------------------------------

model.addConstr(
    gp.quicksum(
        x[c, p]
        for c in customers
        for p in packages
    ) <= 20,
    name="advertising_spots"
)


# ============================================================
# 11. OBJECTIVE FUNCTION
# ============================================================

# Maximize expected profit:
#
# Revenue from the package
# minus
# Expected underdelivery penalty

model.setObjective(
    gp.quicksum(
        expected_profit[c, p] * x[c, p]
        for c in customers
        for p in packages
    ),
    GRB.MAXIMIZE
)


# ============================================================
# 12. SOLVE MODEL
# ============================================================

model.optimize()


# ============================================================
# 13. DISPLAY RESULTS
# ============================================================

if model.status == GRB.OPTIMAL:

    print("\nOPTIMAL ADVERTISING PLAN")
    print("=" * 75)

    total_revenue = 0
    total_expected_penalty = 0

    for c in customers:
        for p in packages:

            if x[c, p].X > 0.5:

                revenue = price[p]
                penalty = expected_penalty[c, p]
                profit = expected_profit[c, p]

                total_revenue += revenue
                total_expected_penalty += penalty

                print(
                    f"{c}: "
                    f"{p.capitalize():12} | "
                    f"Views: {views[p]:>9,} | "
                    f"CPM: ${cpm[p]:>5.2f} | "
                    f"Revenue: ${revenue:>9,.0f} | "
                    f"Expected Penalty: ${penalty:>9,.2f} | "
                    f"Expected Profit: ${profit:>9,.2f}"
                )

    print("=" * 75)

    print(f"Total Revenue:       ${total_revenue:,.2f}")
    print(f"Expected Penalties:  ${total_expected_penalty:,.2f}")
    print(f"Expected Profit:     ${model.ObjVal:,.2f}")

else:

    print("No optimal solution found.")

### Changes from the original

Cleanups only. The model, data, and results are the same as the original notebook.

1. **Probabilities:** the setup text now says the 80% / 60% / 45% / 35% figures are "at least" probabilities, and shows how they convert to range probabilities that add up to 100%.
2. **Assumptions stated:** 1.5M for the "below 1.8M" range, and each range valued at its lower end.
3. **`scenarios` renamed to `packages`** (and the loop letter `s` to `p`). These are the packages we sell, not viewer outcomes. The real-data model will have actual scenarios (the Nielsen increases), so this keeps the two apart.
4. **Constraint 3 comment** said "exactly 20" but the rule is `<= 20` (at most 20). The comment now matches the code.
5. **Unused "Underdelivery Factor" column** removed from the prospects table. The code never used it.
6. **C24** renamed from a second "Grocery Store A" to "Grocery Store C", matching Ed's spreadsheet.
7. **Removed unused code:** `import math`, the repeated `gurobipy` imports, and the `prospects_list.csv` cell (the model never used `df_p`, and that file isn't in the repo). Step 2 replaces the typed-in data with data read from `NFL Ad PKG MAIN.xlsx`.
8. Fixed typos in the setup text ("thosuand", "You need maximize").